# Финальная валидация аугментированного датасета

**Вход:** `train_after_stage3.csv`  
**Выход:** `train_after_stage3_validated.csv`

**Режим:** жёсткий — вырезаем весь брак, не добиваем классы обратно до 55.

### Что удаляется:
1. Точные дубликаты (`label` + `text`) — включая оригиналы
2. Невосстановленные маркеры маскировки `<0>`, `<1>`, ... (брак back-translation)
3. Сломанные маркеры в кавычках: `"9>`, `<12"`, `"14"`
4. Потеря всех NER-плейсхолдеров (только для аугментированных)

Оригиналы (`source == 'original'`) по содержанию не фильтруются — только дубли.

## 1. Импорты и загрузка

In [ ]:
import re
import pandas as pd

INPUT_CSV  = "train_after_stage3.csv"
OUTPUT_CSV = "train_after_stage3_validated.csv"
TARGET_COUNT = 55  # целевое кол-во (для справки в отчёте)

df = pd.read_csv(INPUT_CSV)
print(f"Загружено: {len(df)} строк, {df['label'].nunique()} классов")
print("\nИсточники до валидации:")
print(df['source'].value_counts().to_string())

## 2. Регулярные выражения для поиска брака

In [ ]:
# Невосстановленные маркеры маскировки: <0>, <1>, < 12 > и т.д.
UNREST_RE = re.compile(r"<\s*\d+\s*>")

# Сломанные маркеры — NLLB ломает <N> на кавычки: "9>  <12"  "14"
# (открывающая ИЛИ закрывающая угловая скобка рядом с кавычкой и цифрой)
BROKEN_MARKER_RE = re.compile(r'["«»]\s*\d+\s*>|<\s*\d+\s*["«»]')

# NER-плейсхолдеры: [PERSON], [ORGANIZATION], [DATE_TIME] и т.д.
PLACEHOLDER_RE = re.compile(r"\[[A-Z][A-Z_]*(?:\s[A-Z_]*)?\]")

print("Регекспы готовы")

## 3. Анализ ДО удаления (диагностика)

Сначала смотрим что найдётся, ничего не удаляя.

In [ ]:
aug_mask = df['source'] != 'original'

n_dup    = df.duplicated(subset=['label', 'text']).sum()
n_unrest = (aug_mask & df['text'].str.contains(UNREST_RE)).sum()
n_broken = (aug_mask & df['text'].str.contains(BROKEN_MARKER_RE)).sum()
n_ph     = df['text'].apply(lambda t: len(PLACEHOLDER_RE.findall(t)))
n_noph   = (aug_mask & (n_ph == 0)).sum()

print("=== ДИАГНОСТИКА (что будет удалено) ===")
print(f"  Точные дубликаты (label+text):      {n_dup}")
print(f"  Невосстановленные маркеры <N>:      {n_unrest}")
print(f"  Сломанные маркеры в кавычках:       {n_broken}")
print(f"  Потеря всех плейсхолдеров (аугм.):  {n_noph}")
print()

# Дополнительная диагностика — латинские вкрапления (НЕ удаляем, просто смотрим)
bt = df[df['source'] == 'stage3_backtranslation'].copy()
bt_lat = bt['text'].apply(lambda t: len(re.findall(r'[a-zA-Z]{4,}', t)))
print(f"[Справочно] BT-текстов с латиницей (4+ букв, >=5 слов): {(bt_lat >= 5).sum()}/{len(bt)}")
print("           (это признак непереведённых кусков, но не явный брак — не удаляем)")

## 4. Посмотреть примеры брака перед удалением

Убедимся, что фильтры ловят именно мусор, а не нормальные тексты.

In [ ]:
def show_examples(mask, title, n=2):
    sub = df[mask]
    print(f"=== {title} ({len(sub)} шт) ===")
    for _, row in sub.head(n).iterrows():
        print(f"[{row['label'][:45]}]")
        print(row['text'][:300])
        print('-' * 60)
    print()

show_examples(aug_mask & df['text'].str.contains(UNREST_RE), "Невосстановленные маркеры <N>")
show_examples(aug_mask & df['text'].str.contains(BROKEN_MARKER_RE), "Сломанные маркеры в кавычках")
show_examples(aug_mask & (n_ph == 0), "Потеря плейсхолдеров")

## 5. Удаление брака

In [ ]:
n0 = len(df)

# ── 1. Точные дубликаты (включая оригиналы) ──────────────────────────────
df_clean = df.drop_duplicates(subset=['label', 'text'], keep='first').reset_index(drop=True)
removed_dup = n0 - len(df_clean)

# ── 2-4. Жёсткая чистка ТОЛЬКО аугментированных ──────────────────────────
aug = df_clean['source'] != 'original'
ph_count = df_clean['text'].apply(lambda t: len(PLACEHOLDER_RE.findall(t)))

bad_unrest = aug & df_clean['text'].str.contains(UNREST_RE)
bad_broken = aug & df_clean['text'].str.contains(BROKEN_MARKER_RE)
bad_noph   = aug & (ph_count == 0)

bad_total = bad_unrest | bad_broken | bad_noph

df_clean = df_clean[~bad_total].reset_index(drop=True)

# ── Отчёт ────────────────────────────────────────────────────────────────
print("=== УДАЛЕНО ===")
print(f"  {removed_dup:4d} | Точные дубликаты")
print(f"  {int(bad_unrest.sum()):4d} | Невосстановленные маркеры <N>")
print(f"  {int(bad_broken.sum()):4d} | Сломанные маркеры в кавычках")
print(f"  {int(bad_noph.sum()):4d} | Потеря плейсхолдеров")
print(f"  ----")
print(f"  Итого: {n0} → {len(df_clean)} (удалено {n0 - len(df_clean)})")

## 6. Проверка результата

In [ ]:
print("=== ПРОВЕРКА: брака не осталось ===")
bt_clean = df_clean[df_clean['source'] == 'stage3_backtranslation']
print("Невосстановленные <N>:", bt_clean['text'].str.contains(UNREST_RE).sum())
print("Сломанные маркеры:    ", bt_clean['text'].str.contains(BROKEN_MARKER_RE).sum())
print("Без плейсхолдеров:    ", (bt_clean['text'].apply(lambda t: len(PLACEHOLDER_RE.findall(t))) == 0).sum())
print("Дубли (label+text):   ", df_clean.duplicated(subset=['label', 'text']).sum())
print("NaN/пустые:           ", df_clean['text'].isna().sum() + (df_clean['text'].str.strip() == '').sum())
print()

print("Источники после валидации:")
print(df_clean['source'].value_counts().to_string())

In [ ]:
vc = df_clean['label'].value_counts()
print("=== РАСПРЕДЕЛЕНИЕ ПО КЛАССАМ ===")
print(f"min={vc.min()}, max={vc.max()}, median={vc.median()}, классов={len(vc)}")
print()

below = vc[vc < TARGET_COUNT].sort_values()
print(f"Классов < {TARGET_COUNT}: {len(below)}")
for label, cnt in below.items():
    print(f"  {cnt:3d} | {label}")

## 7. Сохранение

In [ ]:
df_clean.to_csv(OUTPUT_CSV, index=False)
print(f"Сохранено: {OUTPUT_CSV}")
print(f"Всего строк: {len(df_clean)}")